[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-06-etl-transformation.ipynb#scrollTo=ee000001)

---
# Day 6 · Data Transformation Patterns & @does
**certified-journeys / hamilton-certified** · Day 6 · ETL Patterns

> **Goal for today:** Build a full ETL pipeline in Hamilton — ingest → clean → transform → export — using `@does` to eliminate repeated logic, `@check_output` to enforce data contracts, and DuckDB as a free in-process SQL source.

In [ ]:
%pip install -q sf-hamilton duckdb

## The ETL Pattern in Hamilton

A Hamilton ETL pipeline has four layers, each a separate module:

```
ingest.py   →   clean.py   →   transform.py   →   export.py
   ↓               ↓               ↓                  ↓
raw Series    validated     ML-ready features     saved files
```

**DuckDB** is used here as the in-process SQL source — the production equivalent is BigQuery, Snowflake, or Redshift. The Hamilton functions are identical regardless; only the `@config.when` branch changes.

Production equivalent note: replace `duckdb.connect()` with `google.cloud.bigquery.Client()` in the prod branch.

In [ ]:
import sys, types, tempfile, os
import numpy as np
import pandas as pd
import duckdb
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns, config, does, check_output

# Set up an in-memory DuckDB database with a sample table
conn = duckdb.connect()
rng = np.random.default_rng(0)
N = 200
sample_df = pd.DataFrame({
    'customer_id':  range(N),
    'age':          rng.integers(18, 80, N).astype(float),
    'spend':        np.where(rng.random(N) < 0.05, -1.0, rng.exponential(80, N)),  # 5% bad rows
    'tenure':       rng.integers(0, 60, N).astype(float),
    'country_code': rng.choice(['US', 'UK', 'CA', None], N),  # some nulls
})
conn.register('customers', sample_df)
print('DuckDB table ready:', conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0], 'rows')
print('Bad spend rows (spend < 0):', (sample_df['spend'] < 0).sum())
print('Null country_code rows:     ', sample_df['country_code'].isna().sum())

## Step 1 · Ingest Layer — `@config.when` for Multiple Sources

The ingest layer uses `@config.when` to support multiple data sources. Downstream functions see a single `raw_customer_data` node regardless of which source is active.

In [ ]:
# ============================================================
# MODULE: ingest.py
# ============================================================

@config.when(source='duckdb')
def raw_customer_data__duckdb(db_conn: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    """Read from DuckDB. Production equivalent: BigQuery or Snowflake."""
    return db_conn.execute('SELECT * FROM customers').df()

@config.when(source='csv')
def raw_customer_data__csv(csv_path: str) -> pd.DataFrame:
    """Read from a CSV file."""
    return pd.read_csv(csv_path)

ingest_module = types.ModuleType('ingest')
ingest_module.raw_customer_data__duckdb = raw_customer_data__duckdb
ingest_module.raw_customer_data__csv    = raw_customer_data__csv
sys.modules['ingest'] = ingest_module

print('Ingest module defined. Supports: duckdb, csv')

## Step 2 · Clean Layer — `@check_output` for Data Contracts

`@check_output` adds runtime validation to a node's output. If the check fails, Hamilton raises an exception before the bad data flows downstream.

```python
@check_output(data_type=pd.Series, range=(0, 120), allow_nans=False)
def age_cleaned(age: pd.Series) -> pd.Series:
    ...
```

Built-in checks: `data_type`, `range`, `allow_nans`, `max_length`. You can also pass custom validator functions.

In [ ]:
from hamilton.function_modifiers import check_output

# ============================================================
# MODULE: clean.py
# ============================================================

@extract_columns('customer_id', 'age', 'spend', 'tenure', 'country_code')
def raw_columns(raw_customer_data: pd.DataFrame) -> pd.DataFrame:
    """Extract raw columns from ingested DataFrame."""
    return raw_customer_data[['customer_id', 'age', 'spend', 'tenure', 'country_code']]

@check_output(data_type=pd.Series, allow_nans=False)
def age_cleaned(age: pd.Series) -> pd.Series:
    """Clamp age to [18, 100] and fill nulls with median."""
    return age.clip(lower=18, upper=100).fillna(age.median())

@check_output(data_type=pd.Series, allow_nans=False)
def spend_cleaned(spend: pd.Series) -> pd.Series:
    """Replace negative spend with 0 (data entry errors)."""
    return spend.clip(lower=0)

@check_output(data_type=pd.Series, allow_nans=False)
def country_code_cleaned(country_code: pd.Series) -> pd.Series:
    """Fill null country codes with 'UNKNOWN'."""
    return country_code.fillna('UNKNOWN')

@check_output(data_type=pd.Series, allow_nans=False)
def tenure_cleaned(tenure: pd.Series) -> pd.Series:
    """Clamp tenure to [0, 120] months."""
    return tenure.clip(lower=0, upper=120).fillna(0)

clean_module = types.ModuleType('clean')
for fn in [raw_columns, age_cleaned, spend_cleaned, country_code_cleaned, tenure_cleaned]:
    setattr(clean_module, fn.__name__, fn)
sys.modules['clean'] = clean_module

print('Clean module defined with @check_output on all outputs.')

## Step 3 · `@does` — Eliminating Repeated Logic

`@does` delegates a node's implementation to another function. It's Hamilton's DRY tool for when multiple nodes share identical logic but different names.

**Without `@does`:**
```python
def age_normalized(age_cleaned):    return (age_cleaned - age_cleaned.mean()) / age_cleaned.std()
def spend_normalized(spend_cleaned): return (spend_cleaned - spend_cleaned.mean()) / spend_cleaned.std()
def tenure_normalized(tenure_cleaned): return (tenure_cleaned - tenure_cleaned.mean()) / tenure_cleaned.std()
```

**With `@does`:**
```python
def _zscore(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / series.std()

@does(_zscore)
def age_normalized(age_cleaned: pd.Series) -> pd.Series: ...

@does(_zscore)
def spend_normalized(spend_cleaned: pd.Series) -> pd.Series: ...
```

Change the normalization logic once in `_zscore` — all three nodes update automatically.

In [ ]:
from hamilton.function_modifiers import does

# ============================================================
# MODULE: transform.py
# ============================================================

# Shared implementation — not a Hamilton node itself
def _zscore(series: pd.Series) -> pd.Series:
    std = series.std()
    return (series - series.mean()) / (std if std > 0 else 1.0)

# Three nodes that all delegate to the same _zscore logic
@tag(feature_type='numerical')
@does(_zscore)
def age_normalized(age_cleaned: pd.Series) -> pd.Series:
    """Z-scored age."""
    ...

@tag(feature_type='numerical')
@does(_zscore)
def spend_normalized(spend_cleaned: pd.Series) -> pd.Series:
    """Z-scored spend."""
    ...

@tag(feature_type='numerical')
@does(_zscore)
def tenure_normalized(tenure_cleaned: pd.Series) -> pd.Series:
    """Z-scored tenure."""
    ...

@tag(feature_type='boolean')
def is_high_value(spend_normalized: pd.Series) -> pd.Series:
    """True if normalized spend > 1 std dev above mean."""
    return (spend_normalized > 1.0).astype(float)

@tag(feature_type='numerical')
def clv_proxy(spend_cleaned: pd.Series, tenure_cleaned: pd.Series) -> pd.Series:
    """Customer Lifetime Value proxy: spend × tenure months."""
    return spend_cleaned * tenure_cleaned

transform_module = types.ModuleType('transform')
for fn in [age_normalized, spend_normalized, tenure_normalized, is_high_value, clv_proxy]:
    setattr(transform_module, fn.__name__, fn)
sys.modules['transform'] = transform_module

print('Transform module defined. @does used for zscore on age, spend, tenure.')

## Step 4 · Execute the Full ETL Pipeline

In [ ]:
from hamilton.plugins import h_pandas

dr_etl = (
    driver.Builder()
    .with_modules(ingest_module, clean_module, transform_module)
    .with_config({'source': 'duckdb'})
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

OUTPUT_FEATURES = [
    'age_normalized', 'spend_normalized', 'tenure_normalized',
    'is_high_value', 'clv_proxy',
]

result_df = dr_etl.execute(OUTPUT_FEATURES, inputs={'db_conn': conn})

print('ETL pipeline output shape:', result_df.shape)
print('\nNo negative spend values (cleaned):',
      (result_df['spend_normalized'] < result_df['spend_normalized'].min()).sum() == 0)
print('\nResult sample:')
print(result_df.head(5).round(3).to_string())

### What just happened?
- **`@config.when(source='duckdb')`** activated the DuckDB ingest branch — no if/else in any function.
- **`@check_output`** on the clean layer validated each Series before it reached the transform layer — bad spend values were already fixed.
- **`@does(_zscore)`** applied the same normalization logic to three nodes — one implementation, three graph nodes.

## Step 5 · Switch to CSV Source — Zero Code Change

In [ ]:
# Save a CSV, then re-run the exact same pipeline with source='csv'
tmp_csv = tempfile.mktemp(suffix='.csv')
sample_df.to_csv(tmp_csv, index=False)

dr_csv = (
    driver.Builder()
    .with_modules(ingest_module, clean_module, transform_module)
    .with_config({'source': 'csv'})   # only change
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

result_csv = dr_csv.execute(OUTPUT_FEATURES, inputs={'csv_path': tmp_csv})
print('CSV pipeline output shape:', result_csv.shape)

# Verify outputs are identical
pd.testing.assert_frame_equal(result_df.round(8), result_csv.round(8))
print('✓ DuckDB and CSV pipelines produce identical outputs.')

os.remove(tmp_csv)

### What just happened?
- **Switching sources is a single config change** — `source='csv'` instead of `source='duckdb'`.
- **The clean and transform layers are completely unchanged** — they don't know or care about the source.
- **`pd.testing.assert_frame_equal`** confirms the outputs are identical regardless of source — this is a good integration test pattern.

## Step 6 · Testing the Clean Layer in Isolation

In [ ]:
# Unit tests for the clean layer — no Driver, no DB connection
bad_spend = pd.Series([-10.0, 0.0, 50.0, -1.0, 200.0])
cleaned = spend_cleaned(bad_spend)
assert (cleaned >= 0).all(), 'spend_cleaned should have no negative values'
assert cleaned.tolist() == [0.0, 0.0, 50.0, 0.0, 200.0]
print('✓ spend_cleaned: negatives replaced with 0')

ages_with_nulls = pd.Series([15.0, None, 80.0, 200.0, 25.0])
cleaned_ages = age_cleaned(ages_with_nulls)
assert not cleaned_ages.isna().any(), 'age_cleaned should have no NaNs'
assert cleaned_ages.max() <= 100, 'age_cleaned should be clamped to 100'
assert cleaned_ages.min() >= 18, 'age_cleaned should be clamped to 18'
print('✓ age_cleaned: nulls filled, range clamped to [18, 100]')

countries_with_nulls = pd.Series(['US', None, 'UK', None])
cleaned_countries = country_code_cleaned(countries_with_nulls)
assert not cleaned_countries.isna().any()
assert 'UNKNOWN' in cleaned_countries.values
print('✓ country_code_cleaned: nulls filled with UNKNOWN')

# Test @does delegation — age_normalized should behave identically to _zscore
test_series = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0])
assert abs(age_normalized(test_series).mean()) < 1e-10
assert abs(age_normalized(test_series).std() - 1.0) < 1e-10
print('✓ age_normalized (@does _zscore): mean≈0, std≈1')

print('\nAll clean layer tests passed!')

### What just happened?
- **`@does` functions are fully testable** — even though `age_normalized` delegates to `_zscore`, you can still call `age_normalized(series)` directly in tests.
- **The `@check_output` decorator** runs at execute time inside the Driver — the unit test here separately verifies the clean logic using plain assertions.

In [ ]:
# Challenge: add a 4th normalization node — `tenure_years_normalized`
# It should first convert tenure_cleaned from months to years, then z-score it
# Use @does to reuse _zscore
# Verify it produces mean≈0 when tested against a simple input

# def tenure_years(tenure_cleaned: pd.Series) -> pd.Series:
#     ...

# @tag(feature_type='numerical')
# @does(_zscore)
# def tenure_years_normalized(tenure_years: pd.Series) -> pd.Series:
#     ...

print('Add tenure_years_normalized to transform_module and re-run the pipeline!')

---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| ETL = 4 modules | ingest → clean → transform → export, each in its own file |
| `@config.when` for sources | Switch between DuckDB/CSV/API without touching transform logic |
| `@check_output` | Runtime data contract — raises before bad data flows downstream |
| `@does(fn)` | Delegates implementation; change `fn` once, all delegating nodes update |
| `_underscore` functions | Shared helpers — not registered as Hamilton nodes (no `__init__` needed) |
| Source-agnostic clean/transform | The key architectural win: clean and transform layers never see the source type |

> **Tip:** `@does` is Hamilton's DRY tool. Use it when multiple nodes share identical logic but different names — e.g. `normalize_age` and `normalize_income` both call the same scaler.

---
## What's next
**Day 7** → Testing Hamilton pipelines: unit tests for individual functions, `@config.when` branch testing with config injection, and integration tests with `assert_equivalent_dataframes`.

Mark Day 6 complete in your [tracker](../index.html).